In [0]:
# 1. Create widget to receive configuration path

dbutils.widgets.text("path", "")
config_path = dbutils.widgets.get("path")





In [0]:

import json

from pyspark.sql import functions as F
from common_utils.logging import get_logger
from common_utils.scdfunctions import scd_type_1





# 2. Get configuration path
dbutils.widgets.text(
    "path",
    "/Workspace/Users/shashankkotagiri22@gmail.com/Employee-Data-Platform/src/silver/config/config_dimensions.json"
)

config_path = dbutils.widgets.get("path")

logger.info("Reading Silver configuration from %s", config_path)

# 3. Read JSON configuration
with open(config_path, "r") as file:
    config = json.load(file)

# 4. Get dimcounty configuration
dimension_config = config["dimcounty"]

source_table = dimension_config["source_table"]
target_table = dimension_config["target_table"]
key_columns = dimension_config["key_columns"]

# 5. Read Bronze table
logger.info("Reading Bronze table: %s", source_table)

df = spark.table(source_table)

# 6. Transformation function
def transform_dimension(df, dimension_config):

    logger.info("Starting dimension transformation")

    for column in dimension_config["drop_columns"]:
        df = df.drop(column)

    for old_column, new_column in dimension_config["rename_columns"].items():
        df = df.withColumnRenamed(old_column, new_column)

    for column in dimension_config["date_columns"]:
        df = df.withColumn(column, F.to_date(F.col(column)))

    for column in dimension_config["timestamp_columns"]:
        df = df.withColumn(column, F.to_timestamp(F.col(column)))

    for column in dimension_config["yes_no_columns"]:
        df = df.withColumn(
            column,
            F.when(F.col(column) == "Y", "Yes")
             .when(F.col(column) == "N", "No")
             .otherwise(F.col(column))
        )

    logger.info("Dimension transformation completed")

    return df

# 7. Transform dimcounty
dim_county = transform_dimension(df, dimension_config)

# 8. Display transformed data
display(dim_county)

# 9. Create Silver schema
spark.sql("""
    CREATE SCHEMA IF NOT EXISTS employeedataplatform.silver
""")

# 10. Create initial Silver table if it doesn't exist
if not spark.catalog.tableExists(target_table):

    logger.info("Silver table does not exist. Creating initial table")

    dim_county.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(target_table)

    logger.info("Initial Silver table created successfully")

# 11. Perform SCD Type 1
scd_type_1(
    spark,
    dim_county,
    target_table,
    key_columns
)

logger.info("Silver dimcounty processing completed")
